# Model Experimentation — ResNet50 vs HRNet × Frame Count × Bird

Compare DeepLabCut training performance across:

- **Architectures**: ResNet-50, HRNet-W32
- **Training frame counts**: 400, 800, 1400
- **Birds**: Miguel, DavidBowie, Endive

A fixed holdout of **200 frames** (per bird) is reserved with a separate seed so RMSE and training-time comparisons remain fair across all 18 runs (2 archs × 3 frame counts × 3 birds).

In [1]:
import os
import glob
import json
import logging
import random
import time
import importlib
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

import sys
sys.path.insert(0, str(Path(".").resolve()))   # so DLCsupport.py is importable

import DLCsupport as dlcs
dlcs = importlib.reload(dlcs)  # pick up recent edits during notebook iteration
from DLCsupport import (
    as_posix_str,
    validate_path_exists,
    find_latest_snapshot,
    load_bodyparts,
    create_combined_project_if_missing,
    build_combined_dataset,
    set_net_type,
    create_and_train,
    ensure_config_scorer_matches_data
)

import deeplabcut
import xrommtools

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

print(f"Seed        : {SEED}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"DeepLabCut  : {deeplabcut.__version__}")

Loading DLC 3.0.0rc13...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)
Seed        : 42
NumPy       : 1.23.5
Pandas      : 2.3.3
DeepLabCut  : 3.0.0rc13


In [ ]:
## 2. Global Constants
# Centralize seeds, frame budgets, architecture names, and safety toggles.

In [2]:
# Workspace root (one level above Code-Testing/)
ROOT = Path("..")

TASK         = "Canari"
EXPERIMENTER = "Tyler"

# Frame counts to evaluate — one combined project is created per (bird, n)
FRAME_COUNTS  = [200, 400, 800, 1400]

# Frames held out for evaluation. Built once per bird with HOLDOUT_SEED
# so the exact same 200 frames are used across every training comparison.
HOLDOUT_N    = 200
HOLDOUT_SEED = 999   # kept different from TRAIN_SEED on purpose
TRAIN_SEED   = SEED  # reproducible training-frame selection

# DLC PyTorch backbone names to compare
ARCHITECTURES = ["resnet_50"]

EPOCHS_SMOKE = 2
EPOCHS_FULL  = 200

# Safety toggles  — flip to True to actually run training / analysis
RUN_TRAINING      = False
RUN_VIDEO_ANALYSIS = False
USE_FULL_EPOCHS   = False

print("Global constants ready.")
print(f"  Frame counts  : {FRAME_COUNTS}")
print(f"  Architectures : {ARCHITECTURES}")
print(f"  Holdout N     : {HOLDOUT_N}  (seed {HOLDOUT_SEED})")
print(f"  Train seed    : {TRAIN_SEED}")
print(f"  Epochs        : {EPOCHS_FULL if USE_FULL_EPOCHS else EPOCHS_SMOKE} ({'FULL' if USE_FULL_EPOCHS else 'SMOKE'})")

Global constants ready.
  Frame counts  : [200, 400, 800, 1400]
  Architectures : ['resnet_50']
  Holdout N     : 200  (seed 999)
  Train seed    : 42
  Epochs        : 2 (SMOKE)


## 3. Per-Bird Configuration

Each bird gets its own cell with all necessary paths.
Update the date-stamped subfolder (e.g. `Canari-Tyler-2026-03-31`) if you create new single-camera projects on a different date.

In [10]:
# ── Miguel ─────────────────────────────────────────────────────────────────
MIGUEL_ROOT = ROOT / "DeepLabCut" / "Miguel"

MIGUEL_CAM1_CONFIG = (
    MIGUEL_ROOT / "TestData" / "Canari_cam1_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)
MIGUEL_CAM2_CONFIG = (
    MIGUEL_ROOT / "TestData" / "Canari_cam2_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)

# Root folder that will contain all Miguel combined-training sub-projects
MIGUEL_COMBINED_BASE = MIGUEL_ROOT / "TestData"

MIGUEL_DATA_PATH    = MIGUEL_ROOT / "trainingdata"
MIGUEL_DATASET_NAME = "Canari"

# Any existing .avi used only to satisfy DLC project-creation API
MIGUEL_DUMMY_VIDEO  = (
    MIGUEL_ROOT / "trainingdata" / "Miguel_20250816_F6"
    / "MiguelF6_20250816_Cam1_short.avi"
)

# Videos used for evaluation (not seen during training)
MIGUEL_TEST_VIDEO_CAM1 = (
    MIGUEL_ROOT / "newdata" / "Miguel20260401Trial6_Cam1"
    / "MiguelF6_20250816_Cam1_short.avi"
)
MIGUEL_TEST_VIDEO_CAM2 = (
    MIGUEL_ROOT / "newdata" / "Miguel20260401Trial6_Cam2"
    / "MiguelF6_20250816_Cam2_short.avi"
)

print("Miguel paths:")
print("  Cam1 config :", MIGUEL_CAM1_CONFIG)
print("  Cam2 config :", MIGUEL_CAM2_CONFIG)
print("  Data path   :", MIGUEL_DATA_PATH)
print("  Dummy video :", MIGUEL_DUMMY_VIDEO)
print("  Test cam1   :", MIGUEL_TEST_VIDEO_CAM1)
print("  Test cam2   :", MIGUEL_TEST_VIDEO_CAM2)

Miguel paths:
  Cam1 config : ..\DeepLabCut\Miguel\TestData\Canari_cam1_training\Canari-Tyler-2026-03-31\config.yaml
  Cam2 config : ..\DeepLabCut\Miguel\TestData\Canari_cam2_training\Canari-Tyler-2026-03-31\config.yaml
  Data path   : ..\DeepLabCut\Miguel\trainingdata
  Dummy video : ..\DeepLabCut\Miguel\trainingdata\Miguel_20250816_F6\MiguelF6_20250816_Cam1_short.avi
  Test cam1   : ..\DeepLabCut\Miguel\newdata\Miguel20260401Trial6_Cam1\MiguelF6_20250816_Cam1_short.avi
  Test cam2   : ..\DeepLabCut\Miguel\newdata\Miguel20260401Trial6_Cam2\MiguelF6_20250816_Cam2_short.avi


In [ ]:
# ── DavidBowie ─────────────────────────────────────────────────────────────
# NOTE: Mirror Miguel's folder tree under DeepLabCut/DavidBowie/.
#       Update the date-stamped subfolder once the single-camera projects exist.
DAVIDBOWIE_ROOT = ROOT / "DeepLabCut" / "DavidBowie"

#DAVIDBOWIE_CAM1_CONFIG = (
#    DAVIDBOWIE_ROOT / "TestData" / "Canari_cam1_training"
##    / "Canari-Tyler-2026-03-31" / "config.yaml"
#)
#DAVIDBOWIE_CAM2_CONFIG = (
#    DAVIDBOWIE_ROOT / "TestData" / "Canari_cam2_training"
#    / "Canari-Tyler-2026-03-31" / "config.yaml"
#)

DAVIDBOWIE_COMBINED_BASE = DAVIDBOWIE_ROOT / "UpdatedModelBuilds"

DAVIDBOWIE_DATA_PATH    = DAVIDBOWIE_ROOT / "trainingdata"
DAVIDBOWIE_DATASET_NAME = "Canari"

# Replace with the actual training video once it exists
DAVIDBOWIE_DUMMY_VIDEO  = (
    DAVIDBOWIE_ROOT / "trainingdata" / "DavidBowie_20250828_F17"
    / "Cam1.avi"
)

DAVIDBOWIE_TEST_VIDEO_CAM1 = (
    DAVIDBOWIE_ROOT / "newdata" / "DavidBowie_20250828_F17"
    / "Cam1.avi"
)
DAVIDBOWIE_TEST_VIDEO_CAM2 = (
    DAVIDBOWIE_ROOT / "newdata" / "DavidBowie_20250828_F17"
    / "Cam2.avi"
)

print("DavidBowie paths:")
print("  Cam1 config :", DAVIDBOWIE_CAM1_CONFIG)
print("  Cam2 config :", DAVIDBOWIE_CAM2_CONFIG)
print("  Data path   :", DAVIDBOWIE_DATA_PATH)
print("  Dummy video :", DAVIDBOWIE_DUMMY_VIDEO)
print("  Test cam1   :", DAVIDBOWIE_TEST_VIDEO_CAM1)
print("  Test cam2   :", DAVIDBOWIE_TEST_VIDEO_CAM2)

DavidBowie paths:
  Cam1 config : ..\DeepLabCut\DavidBowie\TestData\Canari_cam1_training\Canari-Tyler-2026-03-31\config.yaml
  Cam2 config : ..\DeepLabCut\DavidBowie\TestData\Canari_cam2_training\Canari-Tyler-2026-03-31\config.yaml
  Data path   : ..\DeepLabCut\DavidBowie\trainingdata
  Dummy video : ..\DeepLabCut\DavidBowie\trainingdata\DavidBowie_20250828_F17\Cam1.avi
  Test cam1   : ..\DeepLabCut\DavidBowie\newdata\DavidBowie_20250828_F17\Cam1.avi
  Test cam2   : ..\DeepLabCut\DavidBowie\newdata\DavidBowie_20250828_F17\Cam2.avi


In [3]:
pd.read_csv(r"C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\trainingdata\DavidBowie_20250828_F17\2Dpoints_2025_08_25_DavidBowie_Subset_Trial17_UND.csv")

,Cranium_Left_Anterior_cam1_X,Cranium_Left_Anterior_cam1_Y,Cranium_Left_Anterior_cam2_X,Cranium_Left_Anterior_cam2_Y,Cranium_Left_Middle_cam1_X,Cranium_Left_Middle_cam1_Y,Cranium_Left_Middle_cam2_X,Cranium_Left_Middle_cam2_Y,Cranium_Right_Middle_cam1_X,Cranium_Right_Middle_cam1_Y,...,Pelvis_Right_Superior_cam2_X,Pelvis_Right_Superior_cam2_Y,Pelvis_Left_Superior_cam1_X,Pelvis_Left_Superior_cam1_Y,Pelvis_Left_Superior_cam2_X,Pelvis_Left_Superior_cam2_Y,Pelvis_Left_Inferior_cam1_X,Pelvis_Left_Inferior_cam1_Y,Pelvis_Left_Inferior_cam2_X,Pelvis_Left_Inferior_cam2_Y
0,458.178705,310.653161,272.022414,333.438895,431.825733,290.215511,294.997179,311.551985,429.356353,315.154892,...,433.502897,409.490284,202.670821,372.550765,470.637969,404.669939,202.903580,374.525166,474.673509,407.990614
1,455.691259,306.675936,273.032332,328.468907,428.837736,286.209318,295.006098,308.569761,425.888101,311.174031,...,433.007973,408.991514,203.236327,372.796929,470.493365,404.740463,203.389171,374.767359,474.332384,408.248230
2,453.203430,302.322640,273.539348,325.363288,426.348049,283.200661,297.008966,304.592184,423.905861,307.685837,...,433.007973,408.991514,203.791391,373.028166,470.334231,404.820301,203.878344,375.005891,474.013693,408.495935
3,450.715046,298.712393,274.552654,320.022485,423.860251,280.690646,297.018870,301.114493,421.425437,304.691792,...,432.724400,409.276281,204.328154,373.243811,470.146693,404.958177,204.383551,375.243422,473.741515,408.754537
4,449.221200,295.722245,274.348216,317.255769,421.871032,278.181644,299.015979,299.124913,419.934420,301.195149,...,433.005852,410.487270,204.840011,373.444804,469.923734,405.191044,204.911543,375.481462,473.536900,409.038539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1524,365.931284,180.112985,271.392926,191.918854,367.408353,172.108194,311.704583,183.881007,336.678449,183.440924,...,499.366399,419.118518,263.956342,383.632168,532.167856,422.851358,265.188453,387.123532,536.251957,427.122402
1525,365.437784,182.115003,270.891062,193.906451,366.912877,173.608072,310.709118,185.370418,335.487234,184.738355,...,500.356955,419.617713,264.760967,383.792833,533.014435,422.580448,265.496882,386.622120,537.322572,426.146160
1526,364.446060,184.116391,270.388149,196.390297,365.919521,175.107327,309.713716,186.859801,334.197185,186.937428,...,500.355952,420.614016,264.678919,383.691197,534.239029,422.924950,265.866334,387.460149,538.032963,426.857313
1527,362.559140,186.817716,269.102258,198.663150,364.429517,177.106604,309.214535,189.339816,332.807132,189.436749,...,500.851406,420.614354,265.581897,384.227902,535.216755,423.095485,265.861546,388.450733,537.592860,427.896408


In [12]:
# ── Endive ──────────────────────────────────────────────────────────────────
# NOTE: Mirror Miguel's folder tree under DeepLabCut/Endive/.
#       Update the date-stamped subfolder once the single-camera projects exist.
ENDIVE_ROOT = ROOT / "DeepLabCut" / "Endive"

ENDIVE_CAM1_CONFIG = (
    ENDIVE_ROOT / "TestData" / "Canari_cam1_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)
ENDIVE_CAM2_CONFIG = (
    ENDIVE_ROOT / "TestData" / "Canari_cam2_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)

ENDIVE_COMBINED_BASE = ENDIVE_ROOT / "TestData"

ENDIVE_DATA_PATH    = ENDIVE_ROOT / "trainingdata"
ENDIVE_DATASET_NAME = "Canari"

# Replace with the actual training video once it exists
ENDIVE_DUMMY_VIDEO  = (
    ENDIVE_ROOT / "trainingdata" / "Endive_20251606_F32"
    / "EndiveF42_20251606_Cam1_short.avi"
)


ENDIVE_TEST_VIDEO_CAM1 = (
    ENDIVE_ROOT / "newdata" / "Endive_20251606_F32"
    / "EndiveF42_20251606_Cam1_short.avi"
)
ENDIVE_TEST_VIDEO_CAM2 = (
    ENDIVE_ROOT / "newdata" / "Endive_20251606_F32"
    / "EndiveF42_20251606_Cam2_short.avi"
)

print("Endive paths:")
print("  Cam1 config :", ENDIVE_CAM1_CONFIG)
print("  Cam2 config :", ENDIVE_CAM2_CONFIG)
print("  Data path   :", ENDIVE_DATA_PATH)
print("  Dummy video :", ENDIVE_DUMMY_VIDEO)
print("  Test cam1   :", ENDIVE_TEST_VIDEO_CAM1)
print("  Test cam2   :", ENDIVE_TEST_VIDEO_CAM2)

Endive paths:
  Cam1 config : ..\DeepLabCut\Endive\TestData\Canari_cam1_training\Canari-Tyler-2026-03-31\config.yaml
  Cam2 config : ..\DeepLabCut\Endive\TestData\Canari_cam2_training\Canari-Tyler-2026-03-31\config.yaml
  Data path   : ..\DeepLabCut\Endive\trainingdata
  Dummy video : ..\DeepLabCut\Endive\trainingdata\Endive_20251606_F32\EndiveF42_20251606_Cam1_short.avi
  Test cam1   : ..\DeepLabCut\Endive\newdata\Endive_20251606_F32\EndiveF42_20251606_Cam1_short.avi
  Test cam2   : ..\DeepLabCut\Endive\newdata\Endive_20251606_F32\EndiveF42_20251606_Cam2_short.avi


## 4. Bird Registry
Collect all per-bird variables into one dict so the rest of the notebook can iterate without repeating code.

In [13]:
BIRDS = {

    "DavidBowie": {
        "cam1_config"    : DAVIDBOWIE_CAM1_CONFIG,
        "cam2_config"    : DAVIDBOWIE_CAM2_CONFIG,
        "combined_base"  : DAVIDBOWIE_COMBINED_BASE,
        "data_path"      : DAVIDBOWIE_DATA_PATH,
        "dataset_name"   : DAVIDBOWIE_DATASET_NAME,
        "dummy_video"    : DAVIDBOWIE_DUMMY_VIDEO,
        "test_videos"    : [DAVIDBOWIE_TEST_VIDEO_CAM1, DAVIDBOWIE_TEST_VIDEO_CAM2],
    },
    "Endive": {
        "cam1_config"    : ENDIVE_CAM1_CONFIG,
        "cam2_config"    : ENDIVE_CAM2_CONFIG,
        "combined_base"  : ENDIVE_COMBINED_BASE,
        "data_path"      : ENDIVE_DATA_PATH,
        "dataset_name"   : ENDIVE_DATASET_NAME,
        "dummy_video"    : ENDIVE_DUMMY_VIDEO,
        "test_videos"    : [ENDIVE_TEST_VIDEO_CAM1, ENDIVE_TEST_VIDEO_CAM2],
    },
}

print(f"Registered {len(BIRDS)} birds: {list(BIRDS)}")

Registered 2 birds: ['DavidBowie', 'Endive']


## 5. Create Combined Projects

One combined DLC project is created for **each (bird, frame-count) pair** — 9 training projects total.
A separate **holdout project** (200 frames) is also created per bird so evaluation frames are never in any training set.

All 12 projects are stored in `combined_configs`, keyed by `(bird_name, nframes)` or `(bird_name, "holdout")`.

In [32]:
BIRDS = {

    "DavidBowie": {
        #"cam1_config"    : DAVIDBOWIE_CAM1_CONFIG,
        #"cam2_config"    : DAVIDBOWIE_CAM2_CONFIG,
        "combined_base"  : DAVIDBOWIE_COMBINED_BASE,
        "data_path"      : DAVIDBOWIE_DATA_PATH,
        "dataset_name"   : DAVIDBOWIE_DATASET_NAME,
        "dummy_video"    : DAVIDBOWIE_DUMMY_VIDEO,
        "test_videos"    : [DAVIDBOWIE_TEST_VIDEO_CAM1, DAVIDBOWIE_TEST_VIDEO_CAM2],
    }}

In [33]:
# combined_configs[(bird_name, nframes)]   -> Path to config.yaml  (training projects)
# combined_configs[(bird_name, "holdout")] -> Path to config.yaml  (evaluation project)
combined_configs: dict = {}

for bird_name, bird_cfg in BIRDS.items():
    base = bird_cfg["combined_base"]   # e.g. .../Miguel/TestData/
    dummy = bird_cfg["dummy_video"]


    # ── Training projects — one per frame count ───────────────────────────────
    for n in FRAME_COUNTS:
        cfg_path = create_combined_project_if_missing(
            task=TASK,
            experimenter=EXPERIMENTER,
            combined_project_root=base / f"Canari_combined_n{n}",
            dummy_video=dummy,
        )
        combined_configs[(bird_name, n)] = cfg_path

    print(f"{bird_name}: {len(FRAME_COUNTS)} training + 1 holdout projects ready.")

print(f"\nTotal combined configs registered: {len(combined_configs)}")
for key, path in combined_configs.items():
    print(f"  {key[0]:12s}  {'holdout' if key[1] == 'holdout' else f'n={key[1]:>4}'}  ->  {path}")

print("\nNext: run the bodyparts setup cell to write bird-specific bodyparts into each config.")

INFO | Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\config.yaml


Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\videos"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\labeled-data"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\training-datasets"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\dlc-models"
Attempting to create a symbolic link of the video ...
Symlink creation impossible (exFat architecture?): copying the video instead.
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\trainingdata\DavidBowie_20250828_F17\Cam1.a

INFO | Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n400\Canari-Tyler-2026-04-21\config.yaml


Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n400\Canari-Tyler-2026-04-21\videos"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n400\Canari-Tyler-2026-04-21\labeled-data"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n400\Canari-Tyler-2026-04-21\training-datasets"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n400\Canari-Tyler-2026-04-21\dlc-models"
Attempting to create a symbolic link of the video ...
Symlink creation impossible (exFat architecture?): copying the video instead.
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\trainingdata\DavidBowie_20250828_F17\Cam1.a

INFO | Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n800\Canari-Tyler-2026-04-21\config.yaml


Symlink creation impossible (exFat architecture?): copying the video instead.
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\trainingdata\DavidBowie_20250828_F17\Cam1.avi copied to C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n800\Canari-Tyler-2026-04-21\videos\Cam1.avi
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n800\Canari-Tyler-2026-04-21\videos\Cam1.avi
Generated "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n800\Canari-Tyler-2026-04-21\config.yaml"

A new project with name Canari-Tyler-2026-04-21 is created at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n800 and a configurable file (config.yam

INFO | Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n1400\Canari-Tyler-2026-04-21\config.yaml


Generated "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n1400\Canari-Tyler-2026-04-21\config.yaml"

A new project with name Canari-Tyler-2026-04-21 is created at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n1400 and a configurable file (config.yaml) is stored there. Change the parameters in this file to adapt to your project's needs.
 Once you have changed the configuration file, use the function 'extract_frames' to select frames for labeling.
. [OPTIONAL] Use the function 'add_new_videos' to add new videos to your project (at any stage).
DavidBowie: 4 training + 1 holdout projects ready.

Total combined configs registered: 4
  DavidBowie    n= 200  ->  C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\confi

## 5B. Apply Bird-Specific Bodyparts to Configs

This step writes each bird's bodyparts list from `DLCsupport.BIRD_BODYPARTS` into all training project configs for that bird.

Update bodypart lists in `DLCsupport.py` as needed, then run this cell.

In [34]:
APPLY_BIRD_BODYPARTS = True

bodyparts_configs_by_bird = {
    bird_name: [combined_configs[(bird_name, n)] for n in FRAME_COUNTS]
    for bird_name in BIRDS
}

if not APPLY_BIRD_BODYPARTS:
    print("Skipping bodyparts write step. Set APPLY_BIRD_BODYPARTS = True to enable.")
else:
    bodyparts_apply_df = dlcs.apply_bird_bodyparts_to_configs(
        bodyparts_configs_by_bird,
        strict=True,
    )
    display(bodyparts_apply_df)

    failed = bodyparts_apply_df[bodyparts_apply_df["status"] != "ok"]
    if not failed.empty:
        raise RuntimeError(
            "Bodyparts write verification failed for one or more configs. "
            "Check DLCsupport.BIRD_BODYPARTS and rerun."
        )

    print("Bird-specific bodyparts have been written to all training configs.")

,bird,config,status,n_bodyparts
0,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
1,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
2,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
3,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21


Bird-specific bodyparts have been written to all training configs.


## 6. Build Combined Datasets

Build **9 training datasets** (3 birds × 3 frame counts) plus **3 holdout datasets** (one per bird, 200 frames each).

**Holdout strategy**:  
- Holdout frames are sampled with `HOLDOUT_SEED = 999`.  
- Training frames are sampled with `TRAIN_SEED = 42`.  
- Because the seeds differ, overlap between holdout and training pools is negligible. The same holdout frames will be produced identically every time this cell is re-run.

In [36]:
dataset_build_log = []   # records what was (or would be) built

for bird_name, bird_cfg in BIRDS.items():

    for n in FRAME_COUNTS:
        train_cfg = combined_configs[(bird_name, n)]
        print(f"[{bird_name}] Building training dataset (n={n} frames, seed={TRAIN_SEED}) ...")
        build_combined_dataset(
            combined_config     = train_cfg,
            data_path           = bird_cfg["data_path"],
            dataset_name        = f"{bird_cfg['dataset_name']}_n{n}",
            experimenter        = EXPERIMENTER,
            nframes             = n,
            frame_selection_seed= TRAIN_SEED
        )
        dataset_build_log.append({"bird": bird_name, "split": "train", "nframes": n, "config": str(train_cfg)})

print(f"\n{'─'*60}")
print(f"Dataset builds complete: {len([r for r in dataset_build_log if r['split']=='train'])} training  "
      f"+ {len([r for r in dataset_build_log if r['split']=='holdout'])} holdout")
pd.DataFrame(dataset_build_log)[["bird", "split", "nframes"]].to_string(index=False) \
    and print(pd.DataFrame(dataset_build_log)[["bird", "split", "nframes"]].to_string(index=False))

INFO | Frame selection seed set to 42


[DavidBowie] Building training dataset (n=200 frames, seed=42) ...
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO | Frame selection seed set to 42


...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[DavidBowie] Building training dataset (n=400 frames, seed=42) ...
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO | Frame selection seed set to 42


...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[DavidBowie] Building training dataset (n=800 frames, seed=42) ...
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO | Frame selection seed set to 42


...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[DavidBowie] Building training dataset (n=1400 frames, seed=42) ...
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...
...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset

────────────────────────────────────────────────────────────
Dataset builds complete: 4 training  + 0 holdout
      bird split  nframes
DavidBowie train      200
DavidBowie train      400
DavidBowie train      800
DavidBowie train     1400


In [ ]:
image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
dataset_frame_counts = []

for (bird_name, split_key), config_path in sorted(
    combined_configs.items(),
    key=lambda item: (item[0][0], str(item[0][1])),
):
    labeled_dir = Path(config_path).parent / "labeled-data"
    image_paths = [
        p for p in labeled_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    unique_image_files = {
        p.relative_to(labeled_dir).as_posix()
        for p in image_paths
    }

    # Collapse cam1/cam2 pairs to one logical frame ID when possible.
    unique_logical_frames = {
        p.stem.replace("_cam1_", "_cam_").replace("_cam2_", "_cam_")
        for p in image_paths
    }

    dataset_frame_counts.append({
        "bird": bird_name,
        "split": "holdout" if split_key == "holdout" else "train",
        "nframes": HOLDOUT_N if split_key == "holdout" else split_key,
        "unique_image_files": len(unique_image_files),
        "unique_logical_frames": len(unique_logical_frames),
        "labeled_data_dir": str(labeled_dir),
    })

dataset_frame_counts_df = pd.DataFrame(dataset_frame_counts).sort_values(
    ["bird", "split", "nframes"]
)

print(
    dataset_frame_counts_df[
        ["bird", "split", "nframes", "unique_image_files", "unique_logical_frames"]
    ].to_string(index=False)
)

dataset_frame_counts_df

      bird split  nframes  unique_image_files  unique_logical_frames
DavidBowie train      200                 400                    200
DavidBowie train      400                 800                    400
DavidBowie train      800                1600                    800
DavidBowie train     1400                2800                   1400


,bird,split,nframes,unique_image_files,unique_logical_frames,labeled_data_dir
1,DavidBowie,train,200,400,200,C:\Users\Salle-Cineradio\Documents\MachineLear...
2,DavidBowie,train,400,800,400,C:\Users\Salle-Cineradio\Documents\MachineLear...
3,DavidBowie,train,800,1600,800,C:\Users\Salle-Cineradio\Documents\MachineLear...
0,DavidBowie,train,1400,2800,1400,C:\Users\Salle-Cineradio\Documents\MachineLear...


In [14]:
[element for element in unique_image_files if element.__contains__('001')]

['Canari/DavidBowie_20250828_F17_cam2_0019.png',
 'Canari/DavidBowie_20250828_F17_cam1_0011.png',
 'Canari/DavidBowie_20250828_F17_cam2_0012.png',
 'Canari/DavidBowie_20250828_F17_cam2_0018.png',
 'Canari/DavidBowie_20250828_F17_cam1_0010.png',
 'Canari/DavidBowie_20250828_F17_cam1_0012.png',
 'Canari/DavidBowie_20250828_F17_cam1_0019.png',
 'Canari/DavidBowie_20250828_F17_cam2_0013.png',
 'Canari/DavidBowie_20250828_F17_cam1_0013.png',
 'Canari/DavidBowie_20250828_F17_cam2_0011.png',
 'Canari/DavidBowie_20250828_F17_cam1_0016.png',
 'Canari/DavidBowie_20250828_F17_cam2_0016.png',
 'Canari/DavidBowie_20250828_F17_cam2_0010.png',
 'Canari/DavidBowie_20250828_F17_cam1_0018.png']

In [15]:
unique_image_files

{'Canari/DavidBowie_20250828_F17_cam1_0587.png',
 'Canari/DavidBowie_20250828_F17_cam1_0583.png',
 'Canari/DavidBowie_20250828_F17_cam2_0843.png',
 'Canari/DavidBowie_20250828_F17_cam2_1391.png',
 'Canari/DavidBowie_20250828_F17_cam1_0375.png',
 'Canari/DavidBowie_20250828_F17_cam1_0603.png',
 'Canari/DavidBowie_20250828_F17_cam1_0781.png',
 'Canari/DavidBowie_20250828_F17_cam1_0994.png',
 'Canari/DavidBowie_20250828_F17_cam1_0215.png',
 'Canari/DavidBowie_20250828_F17_cam2_0292.png',
 'Canari/DavidBowie_20250828_F17_cam2_0656.png',
 'Canari/DavidBowie_20250828_F17_cam2_0532.png',
 'Canari/DavidBowie_20250828_F17_cam2_0092.png',
 'Canari/DavidBowie_20250828_F17_cam1_1199.png',
 'Canari/DavidBowie_20250828_F17_cam2_0195.png',
 'Canari/DavidBowie_20250828_F17_cam2_0238.png',
 'Canari/DavidBowie_20250828_F17_cam2_0809.png',
 'Canari/DavidBowie_20250828_F17_cam1_0037.png',
 'Canari/DavidBowie_20250828_F17_cam1_1214.png',
 'Canari/DavidBowie_20250828_F17_cam1_0330.png',
 'Canari/DavidBowie_

## 7. Training — ResNet50

Iterates over all **12 combinations** (2 architectures × 3 frame counts × 2 birds).  
Architecture is hot-swapped by patching `net_type` in each project's `config.yaml` before calling `create_training_dataset`.  
Wall-clock training time is recorded per run for comparison.

In [37]:

RUN_TRAINING      = True
USE_FULL_EPOCHS   = True
epochs = EPOCHS_FULL if USE_FULL_EPOCHS else EPOCHS_SMOKE
training_results = []  # list of dicts — one entry per (bird, n, arch) run

if not RUN_TRAINING:
    print("DRY RUN — set RUN_TRAINING = True to execute training.")
    for bird_name in BIRDS:
        for n in FRAME_COUNTS:
            for arch in ARCHITECTURES:
                training_results.append({
                    "bird": bird_name, "nframes": n, "architecture": arch,
                    "epochs": epochs, "trained": False,
                    "train_time_s": None, "latest_snapshot": None,
                    "notes": "dry run",
                })
else:
    # One confirmation per bird, even if multiple model/frame variants are trained.
    review_configs_by_bird = {
        bird_name: [combined_configs[(bird_name, n)] for n in FRAME_COUNTS]
        for bird_name in BIRDS
    }
#    dlcs.require_bodyparts_review_before_training(
#        run_training=RUN_TRAINING,
#        configs_by_bird=review_configs_by_bird,
#    )

    for bird_name in BIRDS:
        for n in FRAME_COUNTS:
            train_cfg = combined_configs[(bird_name, n)]
            for arch in ARCHITECTURES:
                run_label = f"{bird_name} | n={n} | {arch}"
                print(f"\n{'─'*60}")
                print(f"Training: {run_label}")

                # Switch architecture in config.yaml
                set_net_type(train_cfg, arch)

                # Time the full create_training_dataset + train_network cycle
                t0 = time.perf_counter()
                try:
                    deeplabcut.create_training_dataset(train_cfg)


                    latest_snap = deeplabcut.train_network(train_cfg, epochs=epochs)
                    elapsed = time.perf_counter() - t0
                    training_results.append({
                        "bird": bird_name, "nframes": n, "architecture": arch,
                        "epochs": epochs, "trained": True,
                        "train_time_s": round(elapsed, 1),
                        "latest_snapshot": latest_snap,
                        "notes": "",
                    })
                    print(f"  Done in {elapsed/60:.1f} min  ->  {latest_snap}")
                except Exception as exc:
                    elapsed = time.perf_counter() - t0
                    training_results.append({
                        "bird": bird_name, "nframes": n, "architecture": arch,
                        "epochs": epochs, "trained": False,
                        "train_time_s": round(elapsed, 1),
                        "latest_snapshot": None,
                        "notes": str(exc),
                    })
                    print(f"  ERROR: {exc}")

print(f"\n{'─'*60}")
print(f"Training runs recorded: {len(training_results)}")
pd.DataFrame(training_results)[["bird", "nframes", "architecture", "trained", "train_time_s"]]

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead



────────────────────────────────────────────────────────────
Training: DavidBowie | n=200 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n200\\Canari-Tyler-2026-04-21\\labeled-data\\Cam1', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n200\\Canari-Tyler-2026-04-21\\labeled-data\\Canari_n200']
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuild

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead
INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n400\\Canari-Tyler-2026-04-21\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr21\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Training: DavidBowie | n=800 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n800\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n800\\Canari-Tyler-2026-04-21\\labeled

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n800\\Canari-Tyler-2026-04-21\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr21\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Training: DavidBowie | n=1400 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n1400\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n1400\\Canari-Tyler-2026-04-21\\labe

,bird,nframes,architecture,trained,train_time_s
0,DavidBowie,200,resnet_50,False,0.1
1,DavidBowie,400,resnet_50,False,0.2
2,DavidBowie,800,resnet_50,False,0.3
3,DavidBowie,1400,resnet_50,False,0.6


In [18]:



RUN_TRAINING      = True
USE_FULL_EPOCHS   = True
epochs = EPOCHS_FULL if USE_FULL_EPOCHS else EPOCHS_SMOKE
training_results = []  # list of dicts — one entry per (bird, n, arch) run

if not RUN_TRAINING:
    print("DRY RUN — set RUN_TRAINING = True to execute training.")
    for bird_name in BIRDS:
        for n in FRAME_COUNTS:
            for arch in ARCHITECTURES:
                training_results.append({
                    "bird": bird_name, "nframes": n, "architecture": arch,
                    "epochs": epochs, "trained": False,
                    "train_time_s": None, "latest_snapshot": None,
                    "notes": "dry run",
                })
else:
    # One confirmation per bird, even if multiple model/frame variants are trained.
    review_configs_by_bird = {
        bird_name: [combined_configs[(bird_name, n)] for n in FRAME_COUNTS]
        for bird_name in BIRDS
    }
    dlcs.require_bodyparts_review_before_training(
        run_training=RUN_TRAINING,
        configs_by_bird=review_configs_by_bird,
    )

    for bird_name in BIRDS:
        for n in FRAME_COUNTS:
            train_cfg = combined_configs[(bird_name, n)]
            for arch in ARCHITECTURES:
                run_label = f"{bird_name} | n={n} | {arch}"
                print(f"\n{'─'*60}")
                print(f"Training: {run_label}")

                # Switch architecture in config.yaml
                set_net_type(train_cfg, arch)

                # Time the full create_training_dataset + train_network cycle
                t0 = time.perf_counter()
                try:
                    latest_snap = dlcs.create_and_train(
                        config_path   = train_cfg,
                        epochs        = epochs,
                        snapshot_path = None,   # always start fresh per variant
                    )
                    elapsed = time.perf_counter() - t0
                    training_results.append({
                        "bird": bird_name, "nframes": n, "architecture": arch,
                        "epochs": epochs, "trained": True,
                        "train_time_s": round(elapsed, 1),
                        "latest_snapshot": latest_snap,
                        "notes": "",
                    })
                    print(f"  Done in {elapsed/60:.1f} min  ->  {latest_snap}")
                except Exception as exc:
                    elapsed = time.perf_counter() - t0
                    training_results.append({
                        "bird": bird_name, "nframes": n, "architecture": arch,
                        "epochs": epochs, "trained": False,
                        "train_time_s": round(elapsed, 1),
                        "latest_snapshot": None,
                        "notes": str(exc),
                    })
                    print(f"  ERROR: {exc}")

print(f"\n{'─'*60}")
print(f"Training runs recorded: {len(training_results)}")
pd.DataFrame(training_results)[["bird", "nframes", "architecture", "trained", "train_time_s"]]


Review bodyparts per bird before training:

Bird: DavidBowie
  configured bodyparts (21): ['Cranium_Left_Anterior', 'Cranium_Left_Middle', 'Cranium_Right_Middle', 'Cranium_Right_Posterior', 'Beak_Mandible_Left_Posterior', 'Beak_Mandibile_Left_Anterior', 'Beak_Mandible_Right_Anterior', 'Beak_Maxilla_Right_Anterior', 'Beak_Maxilla_Right_Posterior', 'Beak_Maxilla_Left_Anterior', 'Beak_Maxilla_Left_Posterior', 'Tongue_Lower_Palette', 'Glottis', 'Superior_Trachea', 'Keel_Dorsal_Anterior', 'Keel_Dorsal_Posterior', 'Keel_Inferior', 'Pelvis_Right_Inferior', 'Pelvis_Right_Superior', 'Pelvis_Left_Superior', 'Pelvis_Left_Inferior']
  C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\config.yaml
    bodyparts (21): ['Cranium_Left_Anterior', 'Cranium_Left_Middle', 'Cranium_Right_Middle', 'Cranium_Right_Posterior', 'Beak_Mandible_Left_Posterior', 'Beak_Mandibile_Left_Anterior', 'Beak_Mandib

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


All bird confirmations accepted. Training can proceed.

────────────────────────────────────────────────────────────
Training: DavidBowie | n=200 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n200\\Canari-Tyler-2026-04-21\\labeled-data\\Cam1', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n200\\Canari-Tyler-2026-04-21\\labeled-data\\Canari']
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MN

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n200\\Canari-Tyler-2026-04-21\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr21\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Training: DavidBowie | n=400 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n400\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n400\\Canari-Tyler-2026-04-21\\labeled

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n400\\Canari-Tyler-2026-04-21\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr21\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Training: DavidBowie | n=800 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n800\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n800\\Canari-Tyler-2026-04-21\\labeled

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n800\\Canari-Tyler-2026-04-21\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr21\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Training: DavidBowie | n=1400 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n1400\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n1400\\Canari-Tyler-2026-04-21\\labe

,bird,nframes,architecture,trained,train_time_s
0,DavidBowie,200,resnet_50,False,0.2
1,DavidBowie,400,resnet_50,False,0.3
2,DavidBowie,800,resnet_50,False,0.6
3,DavidBowie,1400,resnet_50,False,0.9


## 7B. Create Combined Projects for Augmentation Runs

Create dedicated combined projects for augmentation experiments so model outputs are organized under augmentation-specific project folders.

In [15]:
# Dedicated config registries for augmentation experiments.
augmentation_combined_configs: dict = {}  # Legacy (bird, n) registry
augmentation_variant_configs: dict = {}   # New (bird, n, augmentation) registry

AUGMENTATION_FRAME_COUNTS = [200, 400, 800]
AUGMENTATION_VARIANTS = [
    {
        "name": "flip_lr",
        "modelprefix": "flip_lr",
    },
    {
        "name": "flip_lr_sharp_blur",
        "modelprefix": "flip_lr_sharp_blur",
    },
]

for bird_name, bird_cfg in BIRDS.items():
    base = bird_cfg["combined_base"]
    dummy = bird_cfg["dummy_video"]

    for n in AUGMENTATION_FRAME_COUNTS:
        for variant in AUGMENTATION_VARIANTS:
            variant_name = variant["name"]
            cfg_path = create_combined_project_if_missing(
                task=TASK,
                experimenter=EXPERIMENTER,
                combined_project_root=base / f"Canari_augexp_{variant_name}_n{n}",
                dummy_video=dummy,
            )
            augmentation_variant_configs[(bird_name, n, variant_name)] = cfg_path

        # Keep compatibility with any older references that use (bird, n).
        augmentation_combined_configs[(bird_name, n)] = augmentation_variant_configs[
            (bird_name, n, AUGMENTATION_VARIANTS[0]["name"])
        ]

print(
    f"Created/reused {len(augmentation_variant_configs)} augmentation project configs "
    f"({len(AUGMENTATION_VARIANTS)} variants x {len(AUGMENTATION_FRAME_COUNTS)} frame counts x {len(BIRDS)} birds)."
)
for (bird_name, n, variant_name), path in sorted(augmentation_variant_configs.items()):
    print(f"  {bird_name:12s}  n={n:>4}  aug={variant_name:18s}  ->  {path}")



# Ensure bird-specific bodyparts are written to all augmentation-specific configs.
augmentation_bodyparts_by_bird = {
    bird_name: [
        augmentation_variant_configs[(bird_name, n, variant["name"])]
        for n in AUGMENTATION_FRAME_COUNTS
        for variant in AUGMENTATION_VARIANTS
    ]
    for bird_name in BIRDS
}
augmentation_bodyparts_df = dlcs.apply_bird_bodyparts_to_configs(
    augmentation_bodyparts_by_bird,
    strict=True,
)
display(augmentation_bodyparts_df)


INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n200\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n200\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n400\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n400\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n800\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n800\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\Endive\TestData\Canari_augexp_flip_lr_n200\Canari-Tyler-2026-04-17\config.yam

Created/reused 12 augmentation project configs (2 variants x 3 frame counts x 2 birds).
  DavidBowie    n= 200  aug=flip_lr             ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n200\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 200  aug=flip_lr_sharp_blur  ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n200\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 400  aug=flip_lr             ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n400\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 400  aug=flip_lr_sharp_blur  ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n400\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 800  aug=flip_lr             ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n800\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 800  aug=flip_lr_sharp_blur  ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n800\Canari-Tyler-2026-04-

,bird,config,status,n_bodyparts
0,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
1,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
2,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
3,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
4,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
5,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
6,Endive,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...,ok,24
7,Endive,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...,ok,24
8,Endive,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...,ok,24
9,Endive,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...,ok,24


## 7B. Adding training datasets

In [16]:

# Populate each augmentation-specific project with combined labeled data (same Step 6 pattern).
augmentation_dataset_build_log = []
for bird_name, bird_cfg in BIRDS.items():
    for n in AUGMENTATION_FRAME_COUNTS:
        for variant in AUGMENTATION_VARIANTS:
            variant_name = variant["name"]
            train_cfg = augmentation_variant_configs[(bird_name, n, variant_name)]
            print(
                f"[{bird_name}] Building augmentation dataset "
                f"(aug={variant_name}, n={n} frames, seed={TRAIN_SEED}) ..."
            )
            build_combined_dataset(
                combined_config=train_cfg,
                data_path=bird_cfg["data_path"],
                dataset_name=bird_cfg["dataset_name"],
                experimenter=EXPERIMENTER,
                nframes=n,
                frame_selection_seed=TRAIN_SEED,
            )
            augmentation_dataset_build_log.append({
                "bird": bird_name,
                "augmentation": variant_name,
                "nframes": n,
                "config": str(train_cfg),
            })

print(f"\n{'─'*60}")
print(f"Augmentation dataset builds complete: {len(augmentation_dataset_build_log)}")
if augmentation_dataset_build_log:
    display(pd.DataFrame(augmentation_dataset_build_log))



INFO | Frame selection seed set to 42


[DavidBowie] Building augmentation dataset (aug=flip_lr, n=200 frames, seed=42) ...


UnboundLocalError: local variable 'camera' referenced before assignment

In [17]:
augmentation_variant_configs

{('DavidBowie',
  200,
  'flip_lr'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_n200/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  200,
  'flip_lr_sharp_blur'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_sharp_blur_n200/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  400,
  'flip_lr'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_n400/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  400,
  'flip_lr_sharp_blur'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_sharp_blur_n400/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  800,
  'flip_lr'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_n800/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  800,
  'flip_lr_sharp_blur'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_sharp_blur_n800/Canari-Tyler-2026-04-17/config.yaml'),
 ('Endive',
  200,
  'flip_lr'): W

In [18]:
image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
dataset_frame_counts = []

for (bird_name, split_key, variant_name), config_path in sorted(
    augmentation_variant_configs.items(),
    key=lambda item: (item[0][0], str(item[0][1])),
):
    labeled_dir = Path(config_path).parent / "labeled-data"
    image_paths = [
        p for p in labeled_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    unique_image_files = {
        p.relative_to(labeled_dir).as_posix()
        for p in image_paths
    }

    # Collapse cam1/cam2 pairs to one logical frame ID when possible.
    unique_logical_frames = {
        p.stem.replace("_cam1_", "_cam_").replace("_cam2_", "_cam_")
        for p in image_paths
    }

    dataset_frame_counts.append({
        "bird": bird_name,
        "split": "holdout" if split_key == "holdout" else "train",
        "nframes": HOLDOUT_N if split_key == "holdout" else split_key,
        "variant": variant_name,
        "unique_image_files": len(unique_image_files),
        "unique_logical_frames": len(unique_logical_frames),
        "labeled_data_dir": str(labeled_dir),
    })

dataset_frame_counts_df = pd.DataFrame(dataset_frame_counts).sort_values(
    ["bird", "split", "nframes", "variant"]
)

print(
    dataset_frame_counts_df[
        ["bird", "split", "nframes", "variant", "unique_image_files", "unique_logical_frames"]
    ].to_string(index=False)
)

dataset_frame_counts_df

      bird split  nframes            variant  unique_image_files  unique_logical_frames
DavidBowie train      200            flip_lr                 400                    200
DavidBowie train      200 flip_lr_sharp_blur                 400                    200
DavidBowie train      400            flip_lr                 800                    400
DavidBowie train      400 flip_lr_sharp_blur                 800                    400
DavidBowie train      800            flip_lr                1600                    800
DavidBowie train      800 flip_lr_sharp_blur                1600                    800
    Endive train      200            flip_lr                 400                    200
    Endive train      200 flip_lr_sharp_blur                 400                    200
    Endive train      400            flip_lr                 800                    400
    Endive train      400 flip_lr_sharp_blur                 800                    400
    Endive train      800       

,bird,split,nframes,variant,unique_image_files,unique_logical_frames,labeled_data_dir
0,DavidBowie,train,200,flip_lr,400,200,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
1,DavidBowie,train,200,flip_lr_sharp_blur,400,200,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
2,DavidBowie,train,400,flip_lr,800,400,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
3,DavidBowie,train,400,flip_lr_sharp_blur,800,400,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
4,DavidBowie,train,800,flip_lr,1600,800,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
5,DavidBowie,train,800,flip_lr_sharp_blur,1600,800,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
6,Endive,train,200,flip_lr,400,200,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...
7,Endive,train,200,flip_lr_sharp_blur,400,200,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...
8,Endive,train,400,flip_lr,800,400,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...
9,Endive,train,400,flip_lr_sharp_blur,800,400,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...


## 8. Augmentation Experiment Matrix (200/400/800)

Train two augmentation variants per bird and frame count with explicit `modelprefix` labels:

- `flip_lr`
- `flip_lr_sharp_blur`

This cell runs one bodyparts confirmation per bird, then trains the augmentation matrix.

In [19]:
import shutil

In [26]:
AUGMENTATION_FRAME_COUNTS = [200, 400, 800]
RUN_AUGMENTATION_EXPERIMENT = True
USE_FULL_EPOCHS_AUG = False
AUG_EPOCHS = EPOCHS_FULL if USE_FULL_EPOCHS_AUG else EPOCHS_SMOKE

if "augmentation_variant_configs" not in globals():
    raise RuntimeError(
        "Run Cell 23 first so augmentation_variant_configs is available."
    )

# Keep these fixed for comparability.
AUG_SHUFFLE = 1
AUG_TRAININGSETINDEX = 0
AUG_GPUTOUSE = None

from deeplabcut.utils.auxiliaryfunctions import edit_config



# Use the same augmentation variants that were used to build project folders.
AUGMENTATION_VARIANTS = [
    {
        "name": "flip_lr",
        "modelprefix": "flip_lr",
        "pose_edits": {
            "fliplr": True,
            "symmetric_pairs": [(0, 14), (1, 12), (2, 13), (3, 11), (4, 9), (5, 10)],
            "sharpening": False,
            "blur": False,
        }
    },
    {
        "name": "flip_lr_sharp_blur",
        "modelprefix": "flip_lr_sharp_blur",
        "pose_edits": {
            "fliplr": True,
            "symmetric_pairs": [(0, 14), (1, 12), (2, 13), (3, 11), (4, 9), (5, 10)],
            "sharpening": True,
            "sharpenratio": 0.3,
            "blur": True,
            "blurratio": 0.3
        }
    },
]



def repair_video_set_labeled_data(config_path: Path) -> None:
    project_dir = Path(config_path).parent
    labeled_data_dir = project_dir / "labeled-data"

    # source folder that already has CollectedData
    source_h5 = sorted(labeled_data_dir.glob("*/CollectedData_*.h5"))
    if not source_h5:
        raise FileNotFoundError(f"No CollectedData_*.h5 found under {labeled_data_dir}")
    src_h5 = source_h5[0]
    src_csv = src_h5.with_suffix(".csv")

    with open(config_path, "r", encoding="utf-8") as fh:
        cfg = yaml.safe_load(fh)

    # For each video in video_sets, ensure matching labeled-data/<video_stem>/CollectedData exists
    for video_path in cfg.get("video_sets", {}).keys():
        stem = Path(str(video_path)).stem  # DB17_cam1
        dst_dir = labeled_data_dir / stem
        dst_dir.mkdir(parents=True, exist_ok=True)

        dst_h5 = dst_dir / src_h5.name
        if not dst_h5.exists():
            shutil.copy2(src_h5, dst_h5)

        if src_csv.exists():
            dst_csv = dst_dir / src_csv.name
            if not dst_csv.exists():
                shutil.copy2(src_csv, dst_csv)


def apply_variant_pose_edits(config_path: Path, variant: dict) -> None:
    """Patch pose_cfg after DLC creates the training dataset for this variant."""
    pose_cfg_path, _, _ = deeplabcut.return_train_network_path(
        as_posix_str(config_path),
 #       shuffle=AUG_SHUFFLE,
   #     trainingsetindex=AUG_TRAININGSETINDEX,
        modelprefix=variant["modelprefix"],
    )
    print(f"pose config path: {pose_cfg_path}")
    pose_cfg_path = Path(pose_cfg_path)
    if not pose_cfg_path.exists():
        raise FileNotFoundError(f"pose_cfg.yaml not found at expected path: {pose_cfg_path}")

    edit_config(str(pose_cfg_path), variant["pose_edits"])
    

    with open(pose_cfg_path, "r", encoding="utf-8") as fh:
        pose_cfg = yaml.safe_load(fh)

    if variant["name"] == "flip_lr_sharp_blur":
        if not bool(pose_cfg.get("sharpening", False)) or not bool(pose_cfg.get("blur", False)):
            raise RuntimeError(
                f"Variant {variant['name']} requires both sharpening and blur in pose_cfg."
            )


augmentation_training_results = []

if not RUN_AUGMENTATION_EXPERIMENT:
    print("Dry run: set RUN_AUGMENTATION_EXPERIMENT = True to execute augmentation training.")
else:
    # Confirm bodyparts once per bird for all augmentation-specific configs.
    aug_review_configs_by_bird = {
        bird_name: [
            augmentation_variant_configs[(bird_name, n, variant["name"])]
            for n in AUGMENTATION_FRAME_COUNTS
            for variant in AUGMENTATION_VARIANTS
        ]
        for bird_name in BIRDS
    }
   # dlcs.require_bodyparts_review_before_training(
   #     run_training=True,
   #     configs_by_bird=aug_review_configs_by_bird,
   # )

    for bird_name in BIRDS:
        for n in AUGMENTATION_FRAME_COUNTS:
            # Keep architecture configurable through the existing notebook constant.
            for arch in ARCHITECTURES:
                for variant in AUGMENTATION_VARIANTS:
                    train_cfg = augmentation_variant_configs[(bird_name, n, variant["name"])]
                    run_label = f"{bird_name} | n={n} | {arch} | {variant['name']}"
                    print(f"\n{'─'*60}")
                    print(f"Augmentation training: {run_label}")

                    # Ensure architecture is set in the variant-specific config.
                    set_net_type(train_cfg, arch)

                    t0 = time.perf_counter()
                    try:
                        print('here')
                        repair_video_set_labeled_data(train_cfg)
                        print('here')
                        ensure_config_scorer_matches_data(train_cfg)
                        print('here')
                        deeplabcut.create_training_dataset(as_posix_str(train_cfg))  
                        
                        
                        print('here')
                        
                        apply_variant_pose_edits(train_cfg, variant)
                        # jump out of loop to avoid running multiple trainings while debugging pose edits
                        latest_snap = deeplabcut.train_network(as_posix_str(train_cfg),
                                                               epochs=AUG_EPOCHS,
                                                               snapshot_path=None,
                                                               modelprefix=variant["modelprefix"])
                        

                        
                        elapsed = time.perf_counter() - t0

                        augmentation_training_results.append({
                            "bird": bird_name,
                            "nframes": n,
                            "architecture": arch,
                            "augmentation": variant["name"],
                            "modelprefix": variant["modelprefix"],
                            "train_config": str(train_cfg),
                            "trained": True,
                            "train_time_s": round(elapsed, 1),
                            "latest_snapshot": latest_snap,
                            "notes": "",
                        })
                        print(f"  Done in {elapsed/60:.1f} min  ->  {latest_snap}")
                    except Exception as exc:
                        elapsed = time.perf_counter() - t0
                        augmentation_training_results.append({
                            "bird": bird_name,
                            "nframes": n,
                            "architecture": arch,
                            "augmentation": variant["name"],
                            "modelprefix": variant["modelprefix"],
                            "train_config": str(train_cfg),
                            "trained": False,
                            "train_time_s": round(elapsed, 1),
                            "latest_snapshot": None,
                            "notes": str(exc),
                        })
                        print(f"  ERROR: {exc}")

print(f"\n{'─'*60}")
print(f"Augmentation runs recorded: {len(augmentation_training_results)}")
if augmentation_training_results:
    pd.DataFrame(augmentation_training_results)[
        ["bird", "nframes", "architecture", "augmentation", "trained", "train_time_s", "train_config", "notes"]
    ]
else:
    print("No augmentation runs executed yet.")

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead



────────────────────────────────────────────────────────────
Augmentation training: DavidBowie | n=200 | resnet_50 | flip_lr
here
here
here
  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\TestData\\Canari_augexp_flip_lr_n200\\Canari-Tyler-2026-04-17\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr17\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Augmentation training: DavidBowie | n=200 | resnet_50 | flip_lr_sharp_blur
here
here


INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


here
  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\TestData\\Canari_augexp_flip_lr_sharp_blur_n200\\Canari-Tyler-2026-04-17\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr17\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Augmentation training: DavidBowie | n=400 | resnet_50 | flip_lr
here
here
here


INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead
INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


...

here
  ERROR: Couldn't find any shuffles with trainingsetindex=0, shuffle=1 and modelprefix=flip_lr. Please check that such a shuffle is defined.

────────────────────────────────────────────────────────────
Augmentation training: Endive | n=400 | resnet_50 | flip_lr_sharp_blur
here
here
here


INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\Endive\\TestData\\Canari_augexp_flip_lr_sharp_blur_n400\\Canari-Tyler-2026-04-17\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr17\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Augmentation training: Endive | n=800 | resnet_50 | flip_lr
here
here
here


INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


here
  ERROR: Couldn't find any shuffles with trainingsetindex=0, shuffle=1 and modelprefix=flip_lr. Please check that such a shuffle is defined.

────────────────────────────────────────────────────────────
Augmentation training: Endive | n=800 | resnet_50 | flip_lr_sharp_blur
here
here
here
  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\Endive\\TestData\\Canari_augexp_flip_lr_sharp_blur_n800\\Canari-Tyler-2026-04-17\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr17\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Augmentation runs recorded: 12


### Model Manipulation

How are we able to pull and train specific layers of the model to improve/transfer